# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aishwarya00608/FlyRank_Assignment1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

# Week 02: ML Task Framing

## 1. My Lane as an ML Task
* **Task Type:** Binary Classification (with probability scoring)
* **High-Level Goal:** Predict whether a proposed piece of content for a given search query will rank in the top 10 (Page 1) within 30 days of publication.
* **Downstream Action:** This model feeds the editorial content prioritization queue. Articles predicted with high probability receive full writer resources and internal link prioritization; articles with low predicted probability are flagged for brief revision or dropped before production spend occurs.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or Proxy
* **The Ideal Ground Truth:** Long-term editorial ROI / high organic traffic with high user satisfaction.
* **The Measurable Target/Proxy:** `target_page_one` (Binary: `1` if the URL achieves average organic search position $\le 10$ over days 14–30 post-publish, otherwise `0`).
* **Proxy Risk & Failure Mode:** An article might rank on page 1 for a high-impression, low-intent query (generating clicks without business conversion) or experience temporary ranking fluctuations due to search engine algorithmic tests.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success Metric
* **Primary Metric:** Precision@Top-K (specifically Precision@20% highest scores) alongside PR-AUC (Precision-Recall AUC).
* **Tradeoff Rationale:** Content production has fixed writer and editor capacity. A false positive costs $300–$500 in wasted authoring spend on an article that fails to rank. A false negative only misses an opportunity in an infinite backlog of keywords. Therefore, the system prioritizes high precision over high recall.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import os
import glob
import pandas as pd
import numpy as np

# 1. Locate and load the dataset
# Searches common relative paths for raw csv data in the repo
possible_paths = [
    "../../data/raw/*.csv",
    "../data/*.csv",
    "../../data/*.csv",
    "data/*.csv"
]

csv_files = []
for pattern in possible_paths:
    csv_files.extend(glob.glob(pattern))

if csv_files:
    data_path = csv_files[0]
    print(f"Loading: {data_path}")
    df_raw = pd.read_csv(data_path)
else:
    # Fallback mock representation matching the FlyRank data schema
    print("No CSV found in standard paths; initializing schema template...")
    df_raw = pd.DataFrame({
        "url_slug": ["guide-to-seo", "best-crm-tools", "how-to-bake-sourdough", "python-pandas-tutorial"],
        "keyword": ["seo guide", "crm tools", "sourdough bread", "pandas tutorial"],
        "search_volume": [12000, 4500, 33000, 8100],
        "keyword_difficulty": [68.5, 42.0, 75.0, 31.5],
        "word_count": [2400, 1800, 1500, 3200],
        "avg_rank_30d": [7.2, 14.5, 22.0, 4.8]
    })

# 2. Define Unit of Analysis
# One row = One published content piece / target keyword pair
df_unit = df_raw.copy()

# 3. Derive Target Column (y)
# Proxy: Average rank <= 10 (Page 1)
rank_col = [col for col in df_unit.columns if "rank" in col.lower() or "position" in col.lower()]

if rank_col:
    df_unit["target_page_one"] = (df_unit[rank_col[0]] <= 10).astype(int)
else:
    # Fallback to click/impression volume if rank is not directly present
    metric_col = [col for col in df_unit.columns if any(k in col.lower() for k in ["click", "imp", "traffic"])][0]
    threshold = df_unit[metric_col].median()
    df_unit["target_page_one"] = (df_unit[metric_col] > threshold).astype(int)

# 4. Display confirmation
print(f"Unit of Analysis: 1 row = 1 unique content-keyword opportunity")
print(f"Dataset Dimensions: {df_unit.shape[0]} rows, {df_unit.shape[1]} columns")
print(f"\nTarget Class Distribution:\n{df_unit['target_page_one'].value_counts(normalize=True)}")

df_unit.head()

No CSV found in standard paths; initializing schema template...
Unit of Analysis: 1 row = 1 unique content-keyword opportunity
Dataset Dimensions: 4 rows, 7 columns

Target Class Distribution:
target_page_one
1    0.5
0    0.5
Name: proportion, dtype: float64


,url_slug,keyword,search_volume,keyword_difficulty,word_count,avg_rank_30d,target_page_one
0,guide-to-seo,seo guide,12000,68.5,2400,7.2,1
1,best-crm-tools,crm tools,4500,42.0,1800,14.5,0
2,how-to-bake-sourdough,sourdough bread,33000,75.0,1500,22.0,0
3,python-pandas-tutorial,pandas tutorial,8100,31.5,3200,4.8,1


## 4. The Unit of Analysis
* **One row represents:** One unique `(target_keyword, content_url)` pairing evaluated at a fixed post-publication window (30 days).
* **Key Features:** Content characteristics (word count, structure, readability), market competition (keyword difficulty, search intent), and historical domain authority.
* **Target Label (`target_page_one`):** A binary indicator (`1` or `0`) showing whether the content achieved page-one search placement.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML Beats a Fixed Rule Here
A deterministic rule (such as `word_count > 2000 AND keyword_difficulty < 40`) fails in practice because:
1. **Non-Linear Interactions:** High domain authority can compensate for shorter content, while highly visual queries require fewer words but specific media assets. Static boolean logic cannot model these trade-offs smoothly.
2. **Dynamic Search Baselines:** Search engine evaluation weights shift across topical categories and intent types. A hardcoded threshold breaks across niches, whereas ML learns category-specific decision boundaries and adapts when retrained on fresh ranking data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.